# Analisis Regresi Komparatif: OLS, WLS, GLS, dan Ridge Regression
**Pengujian Asumsi Klasik, Pemodelan Alternatif, dan Evaluasi Metrik RMSE & AIC pada Data Simulasi**

Penulis: Muhamad Hambali

> **Catatan versi notebook ini:** seluruh baris pada `raw_data_simulasi.txt` dipakai (tanpa `n_sample` / pembatasan sampel).
> Semua angka pada notebook ini dihitung ulang dari data penuh, sehingga bisa berbeda dari versi Rmd yang hanya memakai 2.500 baris pertama.

# 1. PENDAHULUAN & EKSPLORASI DATA AWAL

## 1.1 Latar Belakang & Tujuan Analisis

Regresi linier berbasis *Ordinary Least Squares* (OLS) bertumpu pada asumsi klasik Gauss-Markov. Bila seluruh asumsi terpenuhi, estimator OLS bersifat **BLUE**. Pelanggaran yang diuji di sini:

1. **Non-normalitas sisaan** → inferensi ($t$ dan $F$) kurang reliabel.
2. **Multikolinearitas** → variansi penduga koefisien membengkak. Solusi: **Ridge Regression** (penalti $L_2$).
3. **Heteroskedastisitas** ($\operatorname{Var}(\varepsilon_i)\neq\sigma^2$) → OLS tidak efisien. Solusi: **Weighted Least Squares (WLS)**.
4. **Autokorelasi** ($\operatorname{Cov}(\varepsilon_t,\varepsilon_{t-1})\neq 0$) → standar error OLS terbias ke bawah. Solusi: **Generalized Least Squares (GLS)** dengan struktur $\operatorname{AR}(1)$.

**Tujuan:**

a. Menguji asumsi klasik (normalitas, multikolinearitas, heteroskedastisitas, autokorelasi) pada `raw_data_simulasi.txt`.
b. Mengestimasi empat model: **OLS, WLS, GLS, Ridge**.
c. Membandingkan keempatnya dengan **RMSE** dan **AIC**.
d. Menjelaskan makna setiap keluaran secara faktual.

---

## 1.2 Pemuatan Pustaka

Padanan pustaka R → Python yang dipakai:

| Fungsi di R | Padanan di notebook ini |
|---|---|
| `data.table::fread` | `pandas.read_csv` (mesin C) |
| `car::vif` | invers matriks korelasi, `diag(R⁻¹)` |
| `lmtest::bptest` / `dwtest` | implementasi manual (Koenker/studentized BP, statistik DW) |
| `tseries::jarque.bera.test` | `scipy.stats.jarque_bera` |
| `nlme::gls(corAR1)` | GLS-AR(1) via transformasi Prais-Winsten, **O(n)** |
| `glmnet(alpha=0)` / `cv.glmnet` | Ridge bentuk tertutup + 10-fold CV via statistik cukup |

In [ ]:
import time, csv, os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, optimize, ndimage
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda v: f"{v:,.5f}")
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.figsize": (11, 5), "axes.grid": True, "grid.alpha": .25,
                     "axes.spines.top": False, "axes.spines.right": False})
C_DARK, C_RED, C_BLUE = "#2c3e50", "#e74c3c", "#2980b9"

## 1.3 Pembacaan Data (Seluruh Observasi)

Berkas berisi 1.500.000 observasi dan 31 variabel ($X_1$–$X_{30}$ dan $Y$). **Tidak ada pembatasan sampel**: seluruh baris dibaca dan dipakai oleh keempat model, sehingga perbandingan AIC/RMSE tetap adil (data identik).

> **Mengapa GLS tetap bisa dijalankan pada $N=1{,}5$ juta?** `nlme::gls()` di R membentuk matriks kovarians penuh $N\times N$ (±18 TB). Untuk struktur $\operatorname{AR}(1)$, matriks itu tidak perlu dibentuk: cukup transformasi Prais-Winsten $y^*_1=\sqrt{1-\phi^2}\,y_1,\; y^*_t=y_t-\phi\,y_{t-1}$. Seluruh likelihood dihitung dari matriks jumlah-silang berukuran $32\times32$, sehingga memori dan waktunya linier terhadap $N$. Urutan observasi **tidak diacak**, supaya struktur autokorelasi tetap utuh.

In [ ]:
FILE_PATH = "raw_data_simulasi.txt"   # ubah bila lokasi berkas berbeda

# (Opsional) bila dijalankan di Google Colab dan berkas belum ada, tampilkan dialog unggah
if not os.path.exists(FILE_PATH):
    try:
        from google.colab import files
        up = files.upload()
        FILE_PATH = list(up.keys())[0]
    except ImportError:
        raise FileNotFoundError(f"Berkas '{FILE_PATH}' tidak ditemukan di folder kerja: {os.getcwd()}")

# Deteksi pemisah kolom otomatis dari baris pertama (padanan auto-detect fread)
with open(FILE_PATH, "r") as f:
    header_line = f.readline()
cands = {",": header_line.count(","), ";": header_line.count(";"), "\t": header_line.count("\t"), "|": header_line.count("|")}
best = max(cands, key=cands.get)
sep = best if cands[best] > 0 else r"\s+"

t0 = time.time()
df = pd.read_csv(FILE_PATH, sep=sep)            # SELURUH baris dibaca, tanpa nrows
print(f"Pemisah terdeteksi : {sep!r} | waktu baca: {time.time()-t0:.1f} detik")

n_missing = int(df.isna().sum().sum())
if n_missing > 0:
    df = df.dropna().reset_index(drop=True)     # menjaga urutan observasi
xcols = [f"X{i}" for i in range(1, 31)]
assert all(c in df.columns for c in xcols + ["Y"]), "Kolom X1..X30 dan Y harus ada pada berkas."

X = df[xcols].to_numpy(dtype=float)
y = df["Y"].to_numpy(dtype=float)
n, p = X.shape
Xd = np.column_stack([np.ones(n), X])           # matriks desain dengan intersep
n_cols = df.shape[1]
df_head = df[["X1", "X2", "X3", "X4", "X5", "X28", "X29", "X30", "Y"]].head(5).copy()
del df                                           # hemat memori; seluruh data kini ada di X dan y

print(f"Dimensi data yang dianalisis : {n:,} baris dan {n_cols} kolom (semua data dipakai).")
print(f"Jumlah missing value (NA)    : {n_missing}")

**Penjelasan keluaran:** jumlah baris harus sama dengan jumlah observasi pada berkas (tanpa dipotong). `Jumlah missing value = 0` berarti data lengkap dan tidak diperlukan imputasi. Bila ada NA, baris tersebut dibuang tanpa mengacak urutan.

Cuplikan 5 baris pertama:

In [ ]:
display(df_head.style.format("{:.4f}").set_caption("Tabel 1: Cuplikan Lima Baris Pertama Data Simulasi"))

## 1.4 Statistik Deskriptif Variabel Respon

Sebelum pemodelan, sebaran $Y$ diperiksa: tendensi sentral, penyebaran, dan keberadaan pencilan.

In [ ]:
q = np.quantile(y, [0, .25, .5, .75, 1])
desc_df = pd.DataFrame({
    "Statistik": ["Minimum", "Kuartil 1 (Q1)", "Median", "Mean (Rata-rata)", "Kuartil 3 (Q3)", "Maksimum", "Standar Deviasi (SD)", "Skewness", "Kurtosis (excess)"],
    "Nilai": [q[0], q[1], q[2], y.mean(), q[3], q[4], y.std(ddof=1), stats.skew(y), stats.kurtosis(y)]
})
display(desc_df.style.hide(axis="index").format({"Nilai": "{:.4f}"}).set_caption("Tabel 2: Statistik Deskriptif Variabel Respon Y"))

sk = stats.skew(y)
arah = "menceng ke kanan (positively skewed)" if sk > 0.1 else ("menceng ke kiri (negatively skewed)" if sk < -0.1 else "relatif simetris")
print(f"Mean = {y.mean():.4f}, Median = {np.median(y):.4f}, Skewness = {sk:.4f} → distribusi Y {arah}.")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={"width_ratios": [2, 1]})

# Histogram (seluruh data) + kurva densitas empiris dari histogram halus
cnt, edges = np.histogram(y, bins=200, density=True)
mid = (edges[:-1] + edges[1:]) / 2
ax[0].hist(y, bins=70, density=True, color=C_DARK, edgecolor="white", alpha=.85)
ax[0].plot(mid, ndimage.gaussian_filter1d(cnt, 3), color=C_RED, lw=2)
ax[0].set(title="Distribusi Frekuensi & Densitas Respon Y", xlabel="Nilai Y", ylabel="Densitas")

# Boxplot (seluruh data)
ax[1].boxplot(y, vert=True, patch_artist=True, widths=.5,
              boxprops=dict(facecolor="#3498db", color=C_DARK), medianprops=dict(color=C_DARK, lw=2),
              flierprops=dict(marker=".", markerfacecolor=C_RED, markeredgecolor=C_RED, markersize=2, alpha=.3))
ax[1].set(title="Boxplot Variabel Respon Y", ylabel="Nilai Y", xticks=[])
plt.tight_layout(); plt.show()

iqr = q[3] - q[1]
n_out = int(((y < q[1] - 1.5 * iqr) | (y > q[3] + 1.5 * iqr)).sum())
print(f"Jumlah pencilan (di luar 1.5 x IQR): {n_out:,} dari {n:,} observasi ({100*n_out/n:.2f}%)")

**Penjelasan grafik:** histogram menunjukkan bentuk sebaran $Y$ dan garis merah adalah densitas empiris. Ekor yang lebih panjang pada satu sisi menandakan kemencengan. Pada boxplot, garis tengah adalah median, kotak adalah $Q_1$–$Q_3$ (IQR), dan titik merah adalah pencilan di luar $1{,}5\times$IQR. Pencilan yang banyak berpotensi mengganggu normalitas sisaan.

# 2. BAGIAN A: PENGUJIAN ASUMSI KLASIK REGRESI OLS

Model dasar:
$$Y_i=\beta_0+\sum_{j=1}^{30}\beta_jX_{ij}+\varepsilon_i$$
Sisaan $\hat\varepsilon_i=Y_i-\hat Y_i$ menjadi objek uji asumsi.

In [ ]:
def ols_fit(Xd, y):
    beta = np.linalg.solve(Xd.T @ Xd, Xd.T @ y)
    fit = Xd @ beta
    return beta, fit, y - fit

beta_ols, fit_ols, res_ols = ols_fit(Xd, y)
rss_ols = float(res_ols @ res_ols)
print("Model OLS dasar terestimasi pada seluruh", f"{n:,}", "observasi.")

**Penjelasan kode:** koefisien dihitung dengan persamaan normal $\hat\beta=(X'X)^{-1}X'Y$. `fit_ols` adalah $\hat Y_i=X_i\hat\beta$ dan `res_ols` adalah $e_i=Y_i-\hat Y_i$. Keduanya menjadi input seluruh uji diagnostik.

---

## 2.1 Uji Normalitas Sisaan

$H_0$: sisaan berdistribusi normal; $H_1$: tidak normal. $\alpha=0{,}05$, tolak $H_0$ jika $p<\alpha$.

> **Catatan:** Shapiro-Wilk dibatasi $N\le 5000$ (di R maupun SciPy, dan p-value-nya tidak akurat di atas itu). Karena seluruh data dipakai, Shapiro-Wilk diganti dengan **uji D'Agostino-Pearson ($K^2$)**, yang valid untuk $N$ besar. Q-Q plot memakai kuantil dari **seluruh** data.

In [ ]:
mu_r, sd_r = res_ols.mean(), res_ols.std(ddof=1)

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
# Histogram sisaan vs kurva normal
ax[0].hist(res_ols, bins=80, density=True, color="#34495e", edgecolor="white", alpha=.85)
xs = np.linspace(res_ols.min(), res_ols.max(), 500)
ax[0].plot(xs, stats.norm.pdf(xs, mu_r, sd_r), "--", color=C_RED, lw=2)
ax[0].set(title="Histogram Sisaan OLS vs Kurva Normal", xlabel="Sisaan (Residuals)", ylabel="Densitas")

# Q-Q plot dari kuantil seluruh data (grid probabilitas rapat, termasuk ekor)
probs = stats.norm.cdf(np.linspace(-4.8, 4.8, 400))
sample_q = np.quantile(res_ols, probs)
theo_q = stats.norm.ppf(probs)
q1, q3 = np.quantile(res_ols, [.25, .75]); t1, t3 = stats.norm.ppf([.25, .75])
slope = (q3 - q1) / (t3 - t1); intercept = q1 - slope * t1
ax[1].scatter(theo_q, sample_q, s=10, color=C_BLUE, alpha=.7)
ax[1].plot(theo_q, intercept + slope * theo_q, color=C_RED, lw=2)
ax[1].set(title="Normal Q-Q Plot Sisaan OLS", xlabel="Kuantil Teoritis", ylabel="Kuantil Sampel")
plt.tight_layout(); plt.show()

# Uji formal pada seluruh data
jb = stats.jarque_bera(res_ols)
dp = stats.normaltest(res_ols)
ks = stats.kstest(res_ols, "norm", args=(mu_r, sd_r))
normality_table = pd.DataFrame({
    "Metode_Uji": ["Jarque-Bera Test", "D'Agostino-Pearson K² Test", "Kolmogorov-Smirnov Test"],
    "Statistik_Uji": [jb.statistic, dp.statistic, ks.statistic],
    "p_value": [jb.pvalue, dp.pvalue, ks.pvalue],
})
normality_table["Keputusan"] = np.where(normality_table.p_value < .05, "Tolak H0 (Tidak Normal)", "Gagal Tolak H0 (Normal)")
display(normality_table.style.hide(axis="index").format({"Statistik_Uji": "{:.5f}", "p_value": "{:.3e}"}).set_caption("Tabel 3: Hasil Uji Formal Normalitas Sisaan Model OLS"))
print(f"Skewness sisaan = {stats.skew(res_ols):.4f} | Excess kurtosis sisaan = {stats.kurtosis(res_ols):.4f}")

**Cara membaca hasil:**

- **Histogram:** bila puncak empiris lebih tinggi/ramping dan ekor lebih tebal daripada kurva merah putus-putus, sisaan bersifat leptokurtik (*fat tails*).
- **Q-Q plot:** bila titik biru menempel pada garis merah, sisaan normal. Titik yang melengkung menjauh di kedua ujung menandakan *heavy tails*.
- **Tabel 3:** jika $p<0{,}05$ untuk uji-uji tersebut, $H_0$ ditolak dan sisaan **tidak normal**. Pada $N$ sangat besar, uji formal sangat sensitif sehingga penyimpangan kecil pun terdeteksi. Karena itu keputusan harus dibaca bersama Q-Q plot, skewness, dan kurtosis.
- Konsekuensi: pada $N$ besar, teorema limit pusat membuat uji $t/F$ tetap cukup andal, tetapi ketidaknormalan tetap perlu dicatat.

---

## 2.2 Uji Multikolinearitas

$$\text{VIF}_j=\frac{1}{1-R_j^2}$$
Acuan: VIF < 5 aman; 5–10 sedang; > 10 **parah**. Secara komputasi, VIF sama dengan diagonal invers matriks korelasi prediktor.

In [ ]:
corr = np.corrcoef(X, rowvar=False)
vif_vals = np.diag(np.linalg.inv(corr))
vif_df = pd.DataFrame({"Variabel": xcols, "VIF": vif_vals}).sort_values("VIF", ascending=False).reset_index(drop=True)
display(vif_df.head(12).style.hide(axis="index").format({"VIF": "{:.2f}"}).set_caption("Tabel 4: Dua Belas Variabel dengan Nilai VIF Tertinggi"))

n_high_vif = int((vif_df.VIF > 10).sum())
print(f"Jumlah variabel independen dengan VIF > 10 : {n_high_vif} dari {p} variabel.")
print(f"Nilai VIF maksimum tercatat sebesar        : {vif_df.VIF.max():.2f} ({vif_df.Variabel.iloc[0]})")

In [ ]:
# Matriks korelasi Pearson (segitiga atas)
mask = np.tril(np.ones_like(corr, dtype=bool))
cm = np.ma.array(corr, mask=mask)
fig, ax = plt.subplots(figsize=(9, 7.5))
im = ax.imshow(cm, cmap="RdBu", vmin=-1, vmax=1)
ax.set_xticks(range(p)); ax.set_xticklabels(xcols, rotation=90, fontsize=7)
ax.set_yticks(range(p)); ax.set_yticklabels(xcols, fontsize=7)
ax.grid(False); plt.colorbar(im, ax=ax, shrink=.8)
ax.set_title("Matriks Korelasi Pearson Antar Prediktor (X1 - X30)")
plt.tight_layout(); plt.show()

# Pasangan dengan korelasi terkuat (otomatis dari data)
iu = np.triu_indices(p, 1)
pairs = pd.DataFrame({"Var_1": [xcols[i] for i in iu[0]], "Var_2": [xcols[j] for j in iu[1]], "r": corr[iu]})
pairs["|r|"] = pairs.r.abs()
display(pairs.sort_values("|r|", ascending=False).head(8).drop(columns="|r|").style.hide(axis="index").format({"r": "{:.4f}"}).set_caption("Pasangan Prediktor dengan Korelasi Terkuat"))

In [ ]:
vs = vif_df.sort_values("VIF")
fig, ax = plt.subplots(figsize=(8, 9))
ax.barh(vs.Variabel, vs.VIF, color=np.where(vs.VIF > 10, C_RED, "steelblue"))
ax.axvline(10, ls="--", color="red", lw=1.5)
ax.set(title="Nilai VIF untuk Setiap Prediktor (garis putus-putus: batas kritis VIF = 10)", xlabel="Nilai VIF", ylabel="Variabel Prediktor")
plt.tight_layout(); plt.show()

**Cara membaca hasil:**

- VIF sebesar $k$ berarti variansi penduga $\hat\beta_j$ membengkak $k$ kali dibanding bila $X_j$ ortogonal terhadap prediktor lain.
- Pada heatmap, biru tua = korelasi positif kuat, merah tua = negatif kuat. Pasangan dengan $|r|$ mendekati 1 (lihat tabel pasangan di atas) membuat $(X'X)$ hampir singular (*ill-conditioned*).
- Batang merah pada barplot adalah variabel yang melanggar batas VIF = 10. Bila jumlahnya banyak, multikolinearitas dinilai **parah**: koefisien OLS tidak stabil dan standar errornya besar. Solusi: **Ridge Regression**.

---

## 2.3 Uji Heteroskedastisitas

$H_0$: ragam sisaan homogen; $H_1$: heteroskedastis. Uji: *Studentized Breusch-Pagan* (versi Koenker): regresikan $e_i^2$ pada seluruh prediktor, statistik $LM=n R^2_{aux}\sim\chi^2_{30}$.

In [ ]:
# --- Uji Breusch-Pagan (studentized / Koenker) ---
e2 = res_ols ** 2
b_aux, fit_aux, r_aux = ols_fit(Xd, e2)
r2_aux = 1 - (r_aux @ r_aux) / np.sum((e2 - e2.mean()) ** 2)
bp_stat = n * r2_aux
bp_p = stats.chi2.sf(bp_stat, p)

# --- Plot Residuals vs Fitted: hexbin (seluruh data) + tren rata-rata per kelas fitted ---
fig, ax = plt.subplots(figsize=(10, 5.5))
hb = ax.hexbin(fit_ols, res_ols, gridsize=70, cmap="Blues", mincnt=1, bins="log")
plt.colorbar(hb, ax=ax, label="log10(jumlah observasi)")
bins = np.quantile(fit_ols, np.linspace(0, 1, 41))
idx = np.clip(np.digitize(fit_ols, bins) - 1, 0, 39)
trend_x = np.array([fit_ols[idx == k].mean() for k in range(40)])
trend_y = np.array([res_ols[idx == k].mean() for k in range(40)])
ax.axhline(0, color=C_RED, ls="--", lw=1.5)
ax.plot(trend_x, trend_y, color="#e67e22", lw=2.2, label="Tren rata-rata sisaan")
ax.set(title="Plot Residuals vs Fitted Values (Diagnostik Homoskedastisitas)",
       xlabel="Fitted Values (Ŷ)", ylabel="Residuals (e)")
ax.legend(); plt.tight_layout(); plt.show()

bp_table = pd.DataFrame({"Metode_Uji": ["Studentized Breusch-Pagan Test"], "Statistik_BP": [bp_stat], "Derajat_Bebas": [p], "p_value": [bp_p],
    "Keputusan": ["Tolak H0 (Terjadi Heteroskedastisitas)" if bp_p < .05 else "Gagal Tolak H0 (Homoskedastisitas Terpenuhi)"]})
display(bp_table.style.hide(axis="index").format({"Statistik_BP": "{:.4f}", "Derajat_Bebas": "{:d}", "p_value": "{:.5f}"}).set_caption("Tabel 5: Hasil Uji Heteroskedastisitas Breusch-Pagan"))

**Cara membaca hasil:**

- Pada plot, sisaan yang homoskedastis membentuk pita dengan lebar relatif konstan di sekitar nol. Pola corong (melebar/menyempit) menandakan heteroskedastisitas. Garis oranye (rata-rata sisaan per kelas $\hat Y$) yang datar di sekitar nol menandakan spesifikasi linier yang wajar.
- Pada Tabel 5, $p\ge0{,}05$ → gagal tolak $H_0$ (ragam homogen); $p<0{,}05$ → tolak $H_0$. Pada $N$ sangat besar, uji ini sensitif terhadap penyimpangan kecil, jadi pertimbangkan juga besarnya efek yang terlihat pada plot.
- Bila homoskedastisitas terpenuhi, WLS secara teoretis akan mendekati OLS.

---

## 2.4 Uji Autokorelasi Sisaan

$H_0$: $\rho=0$; $H_1$: $\rho>0$ (autokorelasi positif). Statistik Durbin-Watson:
$$DW=\frac{\sum_{t=2}^n(\hat\varepsilon_t-\hat\varepsilon_{t-1})^2}{\sum_{t=1}^n\hat\varepsilon_t^2}\approx2(1-\hat\rho)$$
$DW\approx2$ → tanpa autokorelasi; $DW\ll2$ → autokorelasi positif; $DW\gg2$ → negatif.

> **Catatan:** untuk $N$ sebesar ini, p-value memakai pendekatan normal asimtotik, $\sqrt{n}\,\hat\rho\sim N(0,1)$ (`dwtest` di R juga beralih ke pendekatan normal untuk $N$ besar).

In [ ]:
# --- ACF seluruh sisaan (lag 1..40) ---
def acf_lags(x, nlags=40):
    xc = x - x.mean(); den = xc @ xc
    return np.array([1.0] + [(xc[k:] @ xc[:-k]) / den for k in range(1, nlags + 1)])

acf_vals = acf_lags(res_ols, 40)
ci = 1.96 / np.sqrt(n)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
ax[0].vlines(range(41), 0, acf_vals, color=C_BLUE, lw=2)
ax[0].axhline(0, color="k", lw=.8)
ax[0].axhline(ci, ls="--", color=C_BLUE); ax[0].axhline(-ci, ls="--", color=C_BLUE)
ax[0].set(title="Autocorrelation Function (ACF) Sisaan OLS", xlabel="Lag", ylabel="ACF")

ax[1].plot(range(1, 201), res_ols[:200], "o-", color=C_DARK, ms=3, lw=1)
ax[1].axhline(0, color="red", ls="--")
ax[1].set(title="Runutan Sisaan (200 Observasi Pertama, hanya tampilan)", xlabel="Indeks Observasi (t)", ylabel="Sisaan (e_t)")
plt.tight_layout(); plt.show()

# --- Durbin-Watson ---
dw = float(np.sum(np.diff(res_ols) ** 2) / (res_ols @ res_ols))
rho_emp = 1 - dw / 2
dw_p = stats.norm.sf(rho_emp * np.sqrt(n))     # H1: rho > 0

dw_table = pd.DataFrame({"Metode_Uji": ["Durbin-Watson Test"], "Statistik_DW": [dw], "Estimasi_Rho_Lag1": [rho_emp], "p_value": [dw_p],
    "Keputusan": ["Tolak H0 (Ada Autokorelasi Positif)" if dw_p < .05 else "Gagal Tolak H0 (Bebas Autokorelasi)"]})
display(dw_table.style.hide(axis="index").format({"Statistik_DW": "{:.5f}", "Estimasi_Rho_Lag1": "{:.5f}", "p_value": "{:.3e}"}).set_caption("Tabel 6: Hasil Uji Autokorelasi Durbin-Watson"))
print(f"ACF lag-1 = {acf_vals[1]:.4f}, lag-2 = {acf_vals[2]:.4f}, lag-3 = {acf_vals[3]:.4f}")

**Cara membaca hasil:**

- **ACF:** batang yang melampaui garis batas 95% menandakan autokorelasi signifikan. Peluruhan eksponensial dari lag 1 ke lag berikutnya adalah ciri khas proses $\operatorname{AR}(1)$.
- **Runutan sisaan:** sisaan positif yang cenderung diikuti sisaan positif (dan sebaliknya) membentuk gelombang mulus, bukan *white noise*.
- **Tabel 6:** $DW$ yang jauh di bawah 2 dengan $\hat\rho$ besar dan $p<0{,}05$ berarti autokorelasi positif signifikan. Dampaknya: OLS **tidak efisien** dan standar error terbias ke bawah, sehingga solusinya **GLS-AR(1)**.

---

# 3. BAGIAN B: PEMODELAN REGRESI

Berdasarkan Bagian A, pelanggaran yang relevan adalah multikolinearitas (→ **Ridge**), autokorelasi $\operatorname{AR}(1)$ (→ **GLS**), dan, bergantung pada hasil Breusch-Pagan, heteroskedastisitas (→ **WLS**, tetap diestimasi sebagai pembanding metodologis).

## 3.1 Model 1: Ordinary Least Squares (OLS)

$$\hat\beta_{OLS}=(X'X)^{-1}X'Y$$

In [ ]:
tss_ols = float(np.sum((y - y.mean()) ** 2))
r2_ols = 1 - rss_ols / tss_ols
adj_r2_ols = 1 - (1 - r2_ols) * (n - 1) / (n - p - 1)
rse_ols = np.sqrt(rss_ols / (n - p - 1))

print("Ringkasan Kinerja Model OLS:")
print(f"- Nilai R-squared (R^2)          : {r2_ols:.5f}")
print(f"- Nilai Adjusted R-squared       : {adj_r2_ols:.5f}")
print(f"- Residual Standard Error (RSE)  : {rse_ols:.5f}")
print(f"- Derajat Bebas Sisaan (df)      : {n - p - 1:,}")

**Penjelasan:** $R^2$ adalah proporsi variabilitas $Y$ yang dijelaskan seluruh prediktor, *adjusted* $R^2$ sudah memperhitungkan jumlah parameter, dan RSE adalah simpangan baku rata-rata sisaan. Walaupun $R^2$ bisa tampak baik, koefisien OLS rentan tidak stabil akibat multikolinearitas dan inferensinya tidak sahih akibat autokorelasi.

---

## 3.2 Model 2: Weighted Least Squares (WLS)

Bobot $w_i=1/\hat\sigma_i^2$ diestimasi lewat regresi pembantu:
$$\ln(\hat\varepsilon_i^2)=\alpha_0+\alpha_1\hat Y_i+u_i,\qquad \hat\sigma_i^2=\exp\big(\widehat{\ln\hat\varepsilon_i^2}\big)$$
$$\hat\beta_{WLS}=(X'WX)^{-1}X'WY$$

In [ ]:
# Langkah 1-3: bobot invers-variansi dari regresi pembantu log(e^2) ~ fitted
log_e2 = np.log(np.maximum(res_ols ** 2, np.finfo(float).tiny))
A = np.column_stack([np.ones(n), fit_ols])
a_coef = np.linalg.solve(A.T @ A, A.T @ log_e2)
var_fitted = np.exp(A @ a_coef)
w = 1.0 / var_fitted

# Langkah 4: estimasi WLS
XtWX = Xd.T @ (Xd * w[:, None])
beta_wls = np.linalg.solve(XtWX, Xd.T @ (w * y))
fit_wls = Xd @ beta_wls
res_wls = y - fit_wls
rss_w = float(np.sum(w * res_wls ** 2))                     # RSS terbobot
ybar_w = np.sum(w * y) / np.sum(w)
r2_wls = 1 - rss_w / float(np.sum(w * (y - ybar_w) ** 2))
adj_r2_wls = 1 - (1 - r2_wls) * (n - 1) / (n - p - 1)
rse_wls = np.sqrt(rss_w / (n - p - 1))

print("Ringkasan Kinerja Model WLS:")
print(f"- Nilai R-squared (R^2, terbobot)    : {r2_wls:.5f}")
print(f"- Nilai Adjusted R-squared           : {adj_r2_wls:.5f}")
print(f"- Residual Standard Error (terbobot) : {rse_wls:.5f}")
print(f"- Rasio bobot maks/min               : {w.max()/w.min():.3f}")

**Penjelasan:** bila data homoskedastis, regresi pembantu menghasilkan $\hat\sigma_i^2$ yang hampir seragam sehingga $W\approx cI$. Skalar $c$ saling meniadakan dalam rumus WLS, jadi WLS ≈ OLS. Rasio bobot maks/min yang mendekati 1 menjadi indikasi tersebut. Perhatikan bahwa $R^2$ dan RSE pada WLS dihitung dengan bobot (sama dengan perilaku `summary.lm` di R).

---

## 3.3 Model 3: Generalized Least Squares (GLS) dengan AR(1)

$$\varepsilon_t=\phi\varepsilon_{t-1}+u_t,\quad \operatorname{Var}(\varepsilon)=\sigma^2\Sigma,\ \ \Sigma_{ij}=\phi^{|i-j|}$$
$$\hat\beta_{GLS}=(X'\Sigma^{-1}X)^{-1}X'\Sigma^{-1}Y$$

**Implementasi efisien (tanpa matriks $N\times N$).** Dengan transformasi Prais-Winsten, $\varepsilon'\Sigma^{-1}\varepsilon=\text{RSS}^*(\phi)/(1-\phi^2)$ dan $|\Sigma|=(1-\phi^2)^{n-1}$. Likelihood Gaussian tepat (*exact*) setelah $\beta$ dan $\sigma^2$ diprofilkan keluar:
$$\ell(\phi)=-\frac n2\Big[\ln 2\pi+\ln\frac{\text{RSS}^*(\phi)}{n}+1\Big]+\frac12\ln(1-\phi^2)$$
$\text{RSS}^*(\phi)$ diperoleh dari matriks jumlah-silang $W=[1,X,y]$ (ukuran $32\times32$) yang dihitung **satu kali** dari seluruh data, lalu $\phi$ dicari dengan optimisasi 1-dimensi (metode Maximum Likelihood).

In [ ]:
# Matriks augmentasi W = [1, X, y] ; hitung statistik jumlah-silang SATU KALI dari seluruh data
Wm = np.column_stack([Xd, y])
A0 = Wm[1:].T @ Wm[1:]
A1 = Wm[1:].T @ Wm[:-1]
A2 = Wm[:-1].T @ Wm[:-1]
w1 = Wm[0]
del Wm

def pw_cross(phi):
    # Matriks jumlah-silang data hasil transformasi Prais-Winsten untuk nilai phi tertentu
    return A0 - phi * (A1 + A1.T) + phi ** 2 * A2 + (1 - phi ** 2) * np.outer(w1, w1)

def gls_solve(phi):
    M = pw_cross(phi)
    Mxx, Mxy, Myy = M[:-1, :-1], M[:-1, -1], M[-1, -1]
    beta = np.linalg.solve(Mxx, Mxy)
    rss_star = Myy - Mxy @ beta
    return beta, rss_star

def gls_loglik(phi):
    _, rss_star = gls_solve(phi)
    return -n / 2 * (np.log(2 * np.pi) + np.log(rss_star / n) + 1) + 0.5 * np.log(1 - phi ** 2)

opt = optimize.minimize_scalar(lambda ph: -gls_loglik(ph), bounds=(-0.999, 0.999), method="bounded", options={"xatol": 1e-10})
phi_hat = float(opt.x)
beta_gls, rss_star = gls_solve(phi_hat)
ll_gls = float(gls_loglik(phi_hat))
sigma_u = np.sqrt(rss_star / n)                         # simpangan baku inovasi u_t
sigma_gls = sigma_u / np.sqrt(1 - phi_hat ** 2)         # simpangan baku marginal (setara 'Residual standard error' nlme)
fit_gls = Xd @ beta_gls

ll_ols = -n / 2 * (np.log(2 * np.pi) + np.log(rss_ols / n) + 1)

print("Ringkasan Kinerja Model GLS (AR1, estimasi Maximum Likelihood):")
print(f"- Parameter Autokorelasi AR(1) (Phi)      : {phi_hat:.5f}   (estimasi empiris DW: {rho_emp:.5f})")
print(f"- Residual Standard Error GLS (marginal)  : {sigma_gls:.5f}")
print(f"- Simpangan baku inovasi (u_t)            : {sigma_u:.5f}")
print(f"- Log-Likelihood Model GLS                : {ll_gls:,.2f}")
print(f"- Log-Likelihood Model OLS                : {ll_ols:,.2f}")
print(f"- Kenaikan Log-Likelihood GLS vs OLS      : {ll_gls - ll_ols:,.2f} poin")

**Penjelasan:**

- $\hat\phi$ adalah korelasi serial lag-1 yang ditaksir model. Nilainya semestinya sangat dekat dengan estimasi empiris dari uji Durbin-Watson.
- Kenaikan log-likelihood GLS terhadap OLS menunjukkan seberapa besar model probabilitas bersama membaik setelah kovarians antar observasi berurutan diperhitungkan.
- Estimasi memakai **Maximum Likelihood**. `nlme::gls()` secara bawaan memakai REML. Pada $N$ sebesar ini, perbedaan keduanya dapat diabaikan.

---

## 3.4 Model 4: Ridge Regression (Penalti $L_2$)

$$\min_{\beta}\ \frac1{2n}\sum_{i=1}^n\Big(Y_i-\beta_0-\sum_j\beta_jX_{ij}\Big)^2+\frac{\lambda}{2}\sum_{j=1}^{30}\beta_j^2$$
$$\hat\beta_{Ridge}=(Z'Z/n+\lambda I)^{-1}Z'Y_c/n\quad(\text{pada } X \text{ terstandardisasi } Z)$$

Parameterisasi di atas sama dengan `glmnet(alpha = 0)`: prediktor distandardisasi internal, intersep tidak dipenalti, dan koefisien dikembalikan ke skala asal. Nilai $\lambda$ optimal dicari dengan **10-fold cross-validation** pada barisan $\lambda$ 100 titik (log-spasi, rasio $10^{-4}$) seperti `cv.glmnet`.

> **Efisiensi pada $N=1{,}5$ juta:** Ridge punya solusi bentuk tertutup, dan setiap lipatan cukup memakai statistik cukup ($\sum x x'$, $\sum x$, $\sum xy$, ...) sehingga data tidak perlu dilewati ulang untuk tiap $\lambda$. Hasilnya identik dengan CV biasa. Baris ditetapkan ke lipatan secara acak (`seed = 42`) hanya untuk keperluan CV, sedangkan data model tidak diacak.

In [ ]:
K = 10
Wc = np.column_stack([X, y])                # [X, y], ukuran n x 31
rng = np.random.default_rng(42)
fold_id = rng.permutation(n) % K

S_tot, s_tot, n_tot = Wc.T @ Wc, Wc.sum(0), n
fold_stats = []
for k in range(K):
    Wk = Wc[fold_id == k]
    fold_stats.append((Wk.T @ Wk, Wk.sum(0), Wk.shape[0]))
del Wc

def ridge_from_stats(S, s, m, lam):
    # Solusi ridge glmnet-style dari statistik cukup sebuah himpunan data (m observasi)
    mu = s / m
    C = S / m - np.outer(mu, mu)
    Cxx, cxy = C[:p, :p], C[:p, p]
    sdx = np.sqrt(np.diag(Cxx))
    R = Cxx / np.outer(sdx, sdx); r = cxy / sdx
    b_std = np.linalg.solve(R + lam * np.eye(p), r)
    beta = b_std / sdx
    b0 = mu[p] - mu[:p] @ beta
    return b0, beta, R

# Barisan lambda ala glmnet (alpha=0 -> alpha_efektif 0.001; nobs > nvars -> rasio 1e-4)
mu_all = s_tot / n_tot
C_all = S_tot / n_tot - np.outer(mu_all, mu_all)
sd_all = np.sqrt(np.diag(C_all)[:p])
lam_max = np.max(np.abs(C_all[:p, p] / sd_all)) / 1e-3
lambdas = np.exp(np.linspace(np.log(lam_max), np.log(lam_max * 1e-4), 100))

cv_mse = np.zeros((K, len(lambdas))); fold_n = np.array([f[2] for f in fold_stats])
for k, (Sk, sk, nk) in enumerate(fold_stats):
    Str, str_, ntr = S_tot - Sk, s_tot - sk, n_tot - nk
    Sxx_k, Sxy_k, Syy_k, Sx_k, Sy_k = Sk[:p, :p], Sk[:p, p], Sk[p, p], sk[:p], sk[p]
    for j, lam in enumerate(lambdas):
        b0, beta, _ = ridge_from_stats(Str, str_, ntr, lam)
        sse = Syy_k - 2 * b0 * Sy_k - 2 * beta @ Sxy_k + nk * b0 ** 2 + 2 * b0 * (beta @ Sx_k) + beta @ Sxx_k @ beta
        cv_mse[k, j] = sse / nk

cvm = (cv_mse * fold_n[:, None]).sum(0) / fold_n.sum()
cvse = cv_mse.std(axis=0, ddof=1) / np.sqrt(K)
i_min = int(np.argmin(cvm))
best_lambda = float(lambdas[i_min])
lambda_1se = float(lambdas[np.where(cvm <= cvm[i_min] + cvse[i_min])[0][0]])

print("Hasil Penyetelan Hiperparameter Ridge (10-fold CV):")
print(f"- Lambda Minimum (lambda.min) : {best_lambda:.5f}")
print(f"- Lambda 1-SE (lambda.1se)    : {lambda_1se:.5f}")

fig, ax = plt.subplots(figsize=(9, 5))
ax.errorbar(np.log(lambdas), cvm, yerr=cvse, fmt="o", ms=3, color=C_RED, ecolor="grey", elinewidth=.8, capsize=0)
ax.axvline(np.log(best_lambda), ls="--", color="k"); ax.axvline(np.log(lambda_1se), ls="--", color="k")
ax.set(title="Kurva 10-Fold Cross-Validation Error Ridge Regression", xlabel="log(λ)", ylabel="Mean-Squared Error")
plt.tight_layout(); plt.show()

# Model Ridge final (seluruh data, lambda.min)
b0_ridge, beta_ridge, R_full = ridge_from_stats(S_tot, s_tot, n_tot, best_lambda)
pred_ridge = b0_ridge + X @ beta_ridge

**Cara membaca grafik:**

- **Sumbu X:** $\log(\lambda)$ (semakin ke kanan, penalti makin besar). **Sumbu Y:** MSE rata-rata 10 lipatan, dengan batang galat standar.
- **Garis putus-putus kiri:** $\lambda_{\min}$ (MSE validasi terendah). **Garis kanan:** $\lambda_{1se}$ (model paling sederhana yang masih dalam 1 SE dari minimum). Model final memakai $\lambda_{\min}$.
- Ridge menyusutkan koefisien mendekati nol tetapi **tidak** mengeliminasinya (berbeda dari Lasso), sehingga seluruh 30 variabel tetap ada.

---

## 3.5 Pembuktian Efek Shrinkage: Koefisien OLS vs Ridge

Variabel dengan multikolinearitas paling parah dipilih otomatis (VIF tertinggi) untuk melihat bagaimana penalti $L_2$ meredam ketidakstabilan koefisien.

In [ ]:
top_vars = vif_df.Variabel.head(8).tolist()
pos = {v: i for i, v in enumerate(xcols)}
coef_comp_df = pd.DataFrame({
    "Variabel": top_vars,
    "Nilai_VIF": [vif_df.set_index("Variabel").VIF[v] for v in top_vars],
    "Koefisien_OLS": [beta_ols[pos[v] + 1] for v in top_vars],
    "Koefisien_Ridge": [beta_ridge[pos[v]] for v in top_vars],
})
display(coef_comp_df.style.hide(axis="index").format({"Nilai_VIF": "{:.2f}", "Koefisien_OLS": "{:.4f}", "Koefisien_Ridge": "{:.4f}"})
        .set_caption("Tabel 7: Efek Penyusutan (Shrinkage) Koefisien OLS vs Ridge pada Variabel dengan VIF Tertinggi"))

print(f"Norma-L2 koefisien slope  OLS   : {np.linalg.norm(beta_ols[1:]):.4f}")
print(f"Norma-L2 koefisien slope  Ridge : {np.linalg.norm(beta_ridge):.4f}")

**Penjelasan:** pada pasangan prediktor yang hampir kolinear sempurna, OLS cenderung memberi koefisien besar dengan tanda berlawanan yang saling meniadakan. Ridge menariknya ke nilai yang lebih kecil dan stabil. Norma-$L_2$ koefisien Ridge yang lebih kecil dari OLS adalah bukti ringkas efek *shrinkage*.

---

# 4. BAGIAN C: PERBANDINGAN MODEL BERDASARKAN RMSE DAN AIC

## 4.1 Formulasi Metrik

1. **RMSE** pada skala $Y$ asli: $\sqrt{\frac1n\sum(Y_i-\hat Y_i)^2}$. Semakin kecil, semakin akurat secara in-sample.
2. **AIC** $=2k-2\ln L$:
   - **OLS & WLS:** likelihood Gaussian standar, $k=p+2=32$ (30 slope + intersep + $\sigma$). Untuk WLS, log-likelihood terbobot seperti `logLik.lm`.
   - **GLS:** likelihood penuh dengan determinan $|\Sigma|$, $k=33$ (tambahan $\phi$).
   - **Ridge:** memakai *effective degrees of freedom*:
   $$\operatorname{df}(\lambda)=\operatorname{tr}\big(Z(Z'Z+n\lambda I)^{-1}Z'\big)+1,\quad \text{AIC}_{Ridge}=n\ln2\pi+n\ln\frac{\text{RSS}}{n}+n+2(\operatorname{df}(\lambda)+1)$$

> **Catatan kecil terhadap versi Rmd:** pada $\operatorname{df}(\lambda)$ penalti glmnet berskala $n\lambda$ terhadap $Z'Z$ (karena fungsi objektifnya dibagi $2n$). Notebook ini memakainya secara konsisten. Versi Rmd memakai $\lambda$ tanpa faktor $n$, yang membuat EDF mendekati 31 dan sedikit melebih-lebihkan kompleksitas Ridge.

## 4.2 Perhitungan Matriks Kinerja

In [ ]:
def calc_rmse(actual, predicted):
    return float(np.sqrt(np.mean((actual - predicted) ** 2)))

rmse = {"Ordinary Least Squares (OLS)": calc_rmse(y, fit_ols),
        "Weighted Least Squares (WLS)": calc_rmse(y, fit_wls),
        "Generalized Least Squares (GLS)": calc_rmse(y, fit_gls),
        "Ridge Regression": calc_rmse(y, pred_ridge)}

# AIC
aic_ols = -2 * ll_ols + 2 * (p + 2)

ll_wls = 0.5 * np.sum(np.log(w)) - n / 2 * (np.log(2 * np.pi) + 1 - np.log(n) + np.log(rss_w))
aic_wls = -2 * ll_wls + 2 * (p + 2)

aic_gls = -2 * ll_gls + 2 * (p + 3)

eig_R = np.linalg.eigvalsh(R_full)
edf_ridge = float(np.sum(eig_R / (eig_R + best_lambda)) + 1)       # +1 intersep
rss_ridge = float(np.sum((y - pred_ridge) ** 2))
aic_ridge = n * np.log(2 * np.pi) + n * np.log(rss_ridge / n) + n + 2 * (edf_ridge + 1)

comparison_table = pd.DataFrame({
    "Model_Regresi": list(rmse.keys()),
    "RMSE": list(rmse.values()),
    "AIC": [aic_ols, aic_wls, aic_gls, aic_ridge],
})
comparison_table["Peringkat_RMSE"] = comparison_table.RMSE.rank().astype(int)
comparison_table["Peringkat_AIC"] = comparison_table.AIC.rank().astype(int)
comparison_table = comparison_table[["Model_Regresi", "RMSE", "Peringkat_RMSE", "AIC", "Peringkat_AIC"]]

best_aic_row = comparison_table.AIC.idxmin()
display(comparison_table.style.hide(axis="index").format({"RMSE": "{:.4f}", "AIC": "{:,.2f}"})
        .apply(lambda r: ["background-color:#d4edda;font-weight:bold" if r.name == best_aic_row else "" for _ in r], axis=1)
        .set_caption("Tabel 8: Matriks Perbandingan Kinerja Model Berdasarkan RMSE dan AIC (baris hijau = AIC terbaik)"))
print(f"EDF Ridge (termasuk intersep) = {edf_ridge:.3f} dari maksimum {p + 1} parameter")

**Penjelasan Tabel 8:** baris hijau menandai model dengan AIC terendah. Peringkat 1 pada RMSE biasanya jatuh ke OLS, sebab OLS meminimalkan RSS tak berbobot secara langsung. Perhatikan selisih nilai antar model, bukan hanya urutannya.

## 4.3 Visualisasi Komparasi

In [ ]:
ct = comparison_table.copy()
short = {"Ordinary Least Squares (OLS)": "OLS", "Weighted Least Squares (WLS)": "WLS",
         "Generalized Least Squares (GLS)": "GLS", "Ridge Regression": "Ridge"}
ct["Model"] = ct.Model_Regresi.map(short)

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
r_ = ct.sort_values("RMSE")
b = ax[0].bar(r_.Model, r_.RMSE, color=plt.cm.Blues(np.linspace(.85, .4, 4)), width=.6)
for rect, v in zip(b, r_.RMSE): ax[0].text(rect.get_x() + rect.get_width() / 2, v, f"{v:.4f}", ha="center", va="bottom", fontweight="bold")
ax[0].set_ylim(ct.RMSE.min() * .95, ct.RMSE.max() * 1.05)
ax[0].set(title="Perbandingan Nilai RMSE Antar Model", ylabel="RMSE")

a_ = ct.sort_values("AIC")
b = ax[1].bar(a_.Model, a_.AIC, color=plt.cm.Oranges(np.linspace(.85, .4, 4)), width=.6)
for rect, v in zip(b, a_.AIC): ax[1].text(rect.get_x() + rect.get_width() / 2, v, f"{v:,.1f}", ha="center", va="bottom", fontweight="bold")
span = ct.AIC.max() - ct.AIC.min()
ax[1].set_ylim(ct.AIC.min() - 0.15 * max(span, 1), ct.AIC.max() + 0.15 * max(span, 1))
ax[1].set(title="Perbandingan Nilai AIC Antar Model", ylabel="AIC")
plt.tight_layout(); plt.show()

**Cara membaca grafik:** sumbu X adalah model (diurutkan dari nilai terkecil) dan sumbu Y adalah nilai metrik. Sumbu Y dipotong (tidak mulai dari nol) agar selisih antar model terlihat, jadi bandingkan **angka pada label batang**, bukan tinggi visualnya.

## 4.4 Pembahasan Hasil Komparasi

**1. Mengapa OLS umumnya punya RMSE in-sample terendah?**
OLS meminimalkan $\sum(Y_i-\hat Y_i)^2$, yaitu persis kuadrat RMSE. WLS meminimalkan kuadrat terbobot, GLS meminimalkan $(Y-X\beta)'\Sigma^{-1}(Y-X\beta)$, dan Ridge menambahkan penalti $\lambda\|\beta\|^2$. Karena itu RMSE tak berbobot ketiganya secara teoretis tidak bisa lebih kecil dari OLS.

**2. Mengapa AIC GLS dapat turun jauh?**
Bila sisaan memang berautokorelasi $\operatorname{AR}(1)$ kuat, OLS memperlakukan sisaan seolah independen ($\Sigma=I$) sehingga likelihood-nya buruk. GLS memperhitungkan $|\Sigma|$ dan korelasi antar observasi berurutan, sehingga log-likelihood naik besar. Selisih $\Delta\text{AIC}>10$ sudah dianggap bukti kuat keunggulan model (Burnham & Anderson).

**3. Mengapa WLS mirip OLS?**
Bila Breusch-Pagan tidak menolak homoskedastisitas, bobot WLS hampir seragam sehingga WLS konvergen ke OLS.

**4. Peran Ridge**
RMSE in-sample Ridge sedikit lebih besar dari OLS karena bias penalti, tetapi koefisiennya jauh lebih stabil pada multikolinearitas berat. Keunggulan ini muncul pada generalisasi *out-of-sample*, yang tidak tertangkap oleh metrik in-sample.

# 5. RINGKASAN HASIL (DIHASILKAN OTOMATIS DARI DATA)

Sel berikut merangkum temuan utama langsung dari hasil perhitungan di atas, sehingga angkanya selalu sesuai dengan data penuh.

In [ ]:
best_aic = comparison_table.loc[comparison_table.AIC.idxmin(), "Model_Regresi"]
best_rmse = comparison_table.loc[comparison_table.RMSE.idxmin(), "Model_Regresi"]
d_aic = aic_ols - aic_gls

print("="*78)
print(f"RINGKASAN (N = {n:,} observasi, seluruh data dipakai)")
print("="*78)
print("A. DIAGNOSTIK ASUMSI KLASIK")
print(f"  - Normalitas        : {'TIDAK normal' if jb.pvalue < .05 else 'Normal'} (JB p={jb.pvalue:.2e}; skew={stats.skew(res_ols):.3f}; ex.kurt={stats.kurtosis(res_ols):.3f})")
print(f"  - Multikolinearitas : {n_high_vif} dari {p} variabel VIF>10; VIF maks = {vif_df.VIF.max():.1f} ({vif_df.Variabel.iloc[0]})")
print(f"  - Heteroskedastisitas: {'TERDETEKSI' if bp_p < .05 else 'Tidak terdeteksi'} (BP={bp_stat:.3f}, p={bp_p:.4f})")
print(f"  - Autokorelasi      : {'TERDETEKSI (positif)' if dw_p < .05 else 'Tidak terdeteksi'} (DW={dw:.4f}, rho≈{rho_emp:.4f}, p={dw_p:.2e})")
print("B. KINERJA MODEL")
print(comparison_table.to_string(index=False, formatters={"RMSE": "{:.4f}".format, "AIC": "{:,.2f}".format}))
print("C. KESIMPULAN KOMPARASI")
print(f"  - AIC terbaik  : {best_aic}  (Δ AIC OLS−GLS = {d_aic:,.2f})")
print(f"  - RMSE terbaik : {best_rmse}")
print(f"  - phi GLS = {phi_hat:.4f} ; lambda ridge (min) = {best_lambda:.5f}")

# 6. KESIMPULAN MENYELURUH

1. **Diagnostik asumsi klasik** (lihat ringkasan otomatis di atas): periksa status normalitas, multikolinearitas (jumlah variabel VIF > 10), heteroskedastisitas (Breusch-Pagan), dan autokorelasi $\operatorname{AR}(1)$ (Durbin-Watson). Pada $N$ sebesar ini, uji formal sangat sensitif, sehingga keputusan sebaiknya dibaca bersama grafik diagnostik dan ukuran efeknya.
2. **Evaluasi model:**
   - **OLS** memberi RMSE in-sample terendah secara matematis, tetapi tidak efisien jika terdapat autokorelasi dan tidak stabil jika terdapat multikolinearitas berat.
   - **WLS** hampir identik dengan OLS apabila data homoskedastis.
   - **GLS-AR(1)** menangkap ketergantungan serial dan umumnya unggul pada AIC bila autokorelasi kuat.
   - **Ridge** menstabilkan koefisien variabel kolinear pada nilai $\lambda$ optimal hasil 10-fold CV.
3. **Rekomendasi:**
   - Untuk **inferensi parameter dan pengujian hipotesis**, gunakan **GLS** (menangani autokorelasi).
   - Untuk **stabilitas koefisien/prediksi** di bawah multikolinearitas ekstrem, gunakan **Ridge**.
   - Solusi integratif yang ideal adalah **Regularized GLS**: memutihkan galat dengan transformasi Prais-Winsten lebih dulu, lalu menerapkan Ridge pada data tertransformasi.